# 📂 Build Synthetic Knowledge Base

This notebook uses **`llama3.2`** running locally via **Ollama** to generate a realistic
synthetic knowledge base for **Qafela (قافلة)**, an Egyptian insurance tech company.

### What gets generated:
| Folder | Contents |
|--------|----------|
| `knowledge-base/company/` | About, overview, culture, careers pages |
| `knowledge-base/employees/` | HR-style profiles for each employee |
| `knowledge-base/products/` | Product spec sheets (features, pricing, roadmap) |
| `knowledge-base/contracts/` | Client contracts (one per product per client) |

### Requirements
- [Ollama](https://ollama.com) installed and running
- `llama3.2` pulled: run `ollama pull llama3.2` in your terminal
- No API key needed — runs 100% locally and free!

### Customize it!
Change the `COMPANY` config in **Step 1** to generate a knowledge base for any company.

## Step 1 — Check Ollama is Running

In [9]:
import subprocess
result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
print(result.stdout)
if "llama3.2" not in result.stdout:
    print("⚠️  llama3.2 not found. Run: ollama pull llama3.2")
else:
    print("✅ llama3.2 is ready!")

NAME                         ID              SIZE      MODIFIED      
gpt-oss:20b                  17052f91a42e    13 GB     5 weeks ago      
llama3.2:latest              a80c4f17acd5    2.0 GB    5 weeks ago      
qwen2.5:14b                  7cdf5a0187d5    9.0 GB    6 months ago     
command-r7b-arabic:latest    4bbe3353e56a    5.1 GB    17 months ago    
mistral:latest               f974a74358d6    4.1 GB    17 months ago    

✅ llama3.2 is ready!


## Step 2 — Setup & Company Config

Ollama exposes an **OpenAI-compatible API** at `http://localhost:11434/v1` — same interface,
local model, zero cost.

Edit the `COMPANY` dictionary to define your company. Everything else is generated from this.

In [10]:
import os
import random
from pathlib import Path
from openai import OpenAI

# ── Model config ───────────────────────────────────────────────────────────────
MODEL_ID    = "llama3.2"                    # local Ollama model
OLLAMA_URL  = "http://localhost:11434/v1"   # Ollama's OpenAI-compatible endpoint
MAX_TOKENS  = 1500
TEMPERATURE = 0.7
OVERWRITE     = False  # False = skip files that already exist (re-running a cell will not overwrite them)
REBUILD_TESTS = False  # True = regenerate evaluation/tests.jsonl (Step 10)

random.seed(7)  # reproducible join years, tiers, and contract dates

# Use OpenAI client pointed at Ollama — no real API key needed
client = OpenAI(base_url=OLLAMA_URL, api_key="ollama")

print(f"✅ Model  : {MODEL_ID} (via Ollama)")
print(f"✅ URL    : {OLLAMA_URL}")

# ── Paths ──────────────────────────────────────────────────────────────────────
ROOT          = Path("..").resolve()
KB            = ROOT / "knowledge-base"
COMPANY_DIR   = KB / "company"
EMPLOYEES_DIR = KB / "employees"
PRODUCTS_DIR  = KB / "products"
CONTRACTS_DIR = KB / "contracts"

for d in [COMPANY_DIR, EMPLOYEES_DIR, PRODUCTS_DIR, CONTRACTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Company config — edit this! ────────────────────────────────────────────────
COMPANY = {
    "name":        "Qafela (قافلة)",
    "founded":     2015,
    "founder":     "Mohamed Elbahluan",
    "industry":    "Insurance Technology (InsurTech) in Egypt",
    "hq":          "Cairo, Egypt",
    "description": "قافلة (Qafela) is an Egyptian AI-powered InsurTech company, headquartered in Cairo. The name means caravan. It sells software platforms to insurance companies in Egypt, helping them offer better products at lower cost.",
    "employees":   32,
    "products": [
        {"name": "Carllm",    "type": "Auto Insurance Platform"},
        {"name": "Homellm",   "type": "Home Insurance Platform"},
        {"name": "Healthllm", "type": "Health Insurance Platform"},
        {"name": "Lifellm",   "type": "Life Insurance Platform"},
        {"name": "Bizllm",    "type": "Commercial/Business Insurance Platform"},
        {"name": "Claimllm",  "type": "Claims Processing Platform"},
        {"name": "Rellm",     "type": "Reinsurance Platform"},
        {"name": "Markellm",  "type": "Insurance Marketplace Platform"},
    ],
    "employee_roles": [
        {"name": "Mohamed Elbahluan",  "title": "Co-Founder & CEO",              "dept": "Executive"},
        {"name": "Mariam Hassan",      "title": "Chief Technology Officer",      "dept": "Engineering"},
        {"name": "Ahmed Fathy",        "title": "Chief Financial Officer",       "dept": "Finance"},
        {"name": "Nour El-Din Farouk", "title": "General Counsel",               "dept": "Legal"},
        {"name": "Hana Abdelrahman",   "title": "VP of People",                  "dept": "People"},
        {"name": "Karim Mostafa",      "title": "VP of Sales",                   "dept": "Sales"},
        {"name": "Salma Ibrahim",      "title": "Head of Product",               "dept": "Product"},
        {"name": "Omar Sherif",        "title": "VP of Engineering",              "dept": "Engineering"},
        {"name": "Mohamed Tarek",      "title": "Senior Software Engineer",       "dept": "Engineering"},
        {"name": "Layla Nabil",        "title": "Backend Engineer",               "dept": "Engineering"},
        {"name": "Hassan Mahmoud",     "title": "Frontend Engineer",              "dept": "Engineering"},
        {"name": "Dina Kamal",         "title": "Machine Learning Engineer",      "dept": "Engineering"},
        {"name": "Tarek Soliman",      "title": "DevOps Engineer",                "dept": "Engineering"},
        {"name": "Amr Hegazy",         "title": "Security Engineer",              "dept": "Engineering"},
        {"name": "Yasmin Fouad",       "title": "QA Engineer",                    "dept": "Engineering"},
        {"name": "Nadine Ashraf",      "title": "Data Scientist",                  "dept": "Data"},
        {"name": "Khaled Samir",       "title": "Data Analyst",                    "dept": "Data"},
        {"name": "Farida Adel",        "title": "Product Manager",                 "dept": "Product"},
        {"name": "Mostafa Gamal",      "title": "Product Manager",                 "dept": "Product"},
        {"name": "Aya Ramadan",        "title": "UX Designer",                     "dept": "Design"},
        {"name": "Heba Yassin",        "title": "Product Designer",                "dept": "Design"},
        {"name": "Rania Saeed",        "title": "Marketing Manager",               "dept": "Marketing"},
        {"name": "Mahmoud Zaki",       "title": "Content Marketing Lead",          "dept": "Marketing"},
        {"name": "Sara Lotfy",         "title": "Account Executive",               "dept": "Sales"},
        {"name": "Ibrahim Reda",       "title": "Account Executive",               "dept": "Sales"},
        {"name": "Mona Kamel",         "title": "Sales Engineer",                  "dept": "Sales"},
        {"name": "Hossam Ali",         "title": "Customer Success Manager",        "dept": "Customer Success"},
        {"name": "Noha Farid",         "title": "Customer Success Manager",        "dept": "Customer Success"},
        {"name": "Ali Mansour",        "title": "Support Lead",                    "dept": "Customer Success"},
        {"name": "Ghada Nasser",       "title": "Finance Manager",                 "dept": "Finance"},
        {"name": "Sherif Adel",        "title": "Recruiter",                       "dept": "People"},
        {"name": "Fatma El-Sayed",     "title": "Operations Manager",              "dept": "Operations"},
    ],
    "clients_per_product": 2,   # number of contracts to generate per product
}

print(f"\n✅ Config ready for : {COMPANY['name']}")
print(f"   Products          : {len(COMPANY['products'])}")
print(f"   Employees         : {len(COMPANY['employee_roles'])}")
print(f"   Contracts         : {len(COMPANY['products']) * COMPANY['clients_per_product']} total")

✅ Model  : llama3.2 (via Ollama)
✅ URL    : http://localhost:11434/v1

✅ Config ready for : Qafela (قافلة)
   Products          : 8
   Employees         : 32
   Contracts         : 16 total


## Step 3 — Helper: Call LLM & Save File

We use the standard `openai` Python package pointed at Ollama's local server.
The call is **identical** to calling OpenAI — just a different `base_url`.

In [11]:
def llm(prompt: str, system: str = "You are a helpful business document writer. Write realistic, detailed content.", retries: int = 3) -> str:
    """
    Call llama3.2 running locally via Ollama.
    Uses OpenAI-compatible chat completions API — zero cost, no internet needed.
    """
    for attempt in range(1, retries + 1):
        try:
            response = client.chat.completions.create(
                model=MODEL_ID,
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user",   "content": prompt},
                ],
                max_tokens=MAX_TOKENS,
                temperature=TEMPERATURE,
            )
            break
        except Exception as exc:
            if attempt == retries:
                raise
            print(f"  ⚠️  retry {attempt}/{retries - 1}: {exc}")
    choice = response.choices[0]
    if choice.finish_reason == "length":
        print("  ⚠️  output hit MAX_TOKENS and was cut off — raise MAX_TOKENS")
    return choice.message.content


def save(path: Path, content: str):
    """Write content to a .md file and print confirmation."""
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")
    print(f"  ✅ Saved: {path.relative_to(ROOT)}")


def should_skip(path: Path) -> bool:
    """True if the file exists and OVERWRITE is off."""
    if path.exists() and not OVERWRITE:
        print(f"  ⏭️  Exists, skipped: {path.relative_to(ROOT)}")
        return True
    return False


# ── Quick sanity check ─────────────────────────────────────────────────────────
print("🧪 Testing Ollama connection...")
test = llm("Say exactly: 'Hello from Llama 3.2 via Ollama!' and nothing else.")
print(f"   Response: {test.strip()}")
print("✅ Ollama is working!")

🧪 Testing Ollama connection...
   Response: Hello from Llama 3.2 via Ollama!
✅ Ollama is working!


## Step 4 — Generate Company Pages

Creates 4 company-level documents:
- `about.md` — founding story, mission, growth  
- `overview.md` — products, market position, financials  
- `culture.md` — values, team culture, remote work policy  
- `careers.md` — hiring, benefits, open roles

In [12]:
c = COMPANY  # shorthand
product_list = ", ".join(p["name"] for p in c["products"])

company_pages = {
    "about.md": f"""
Write a detailed 'About Us' page for {c['name']}, an {c['industry']} company.
Founded in {c['founded']} by {c['founder']} in {c['hq']}.
Description: {c['description']}
Products: {product_list}
Current employee count: {c['employees']}

Include: founding story, mission, product history, growth milestones, current state.
Write in markdown with headers. Be specific with dates, numbers, and facts. About 400 words.
Output only the markdown document, nothing else.
""",
    "overview.md": f"""
Write a company overview document for {c['name']}.
Industry: {c['industry']}. HQ: {c['hq']}. Founded: {c['founded']}.
Products: {product_list}

Include: executive summary, product portfolio summary with brief descriptions,
market position, key metrics (contracts, revenue range, client count), and strategic priorities.
Write in markdown with headers. Be specific with plausible business numbers. About 400 words.
Output only the markdown document, nothing else.
""",
    "culture.md": f"""
Write a culture and values document for {c['name']}.
The company is a lean, remote-first {c['industry']} startup with {c['employees']} employees.

Include: core values (at least 5), remote work policy, team rituals, DEI commitments,
work-life balance philosophy, learning and development programs.
Write in markdown with headers. Tone should be authentic and specific. About 350 words.
Output only the markdown document, nothing else.
""",
    "careers.md": f"""
Write a careers page for {c['name']}.
The company has {c['employees']} employees and is a remote-first {c['industry']} company.

Include: why join us, benefits package (salary, equity, health, PTO, L&D budget),
hiring process steps, open roles (at least 5 real-sounding roles with descriptions),
and one employee testimonial.
Write in markdown with headers. Be specific and compelling. About 400 words.
Output only the markdown document, nothing else.
""",
}

print(f"🏢 Generating company pages for {c['name']}...\n")
for filename, prompt in company_pages.items():
    if should_skip(COMPANY_DIR / filename):
        continue
    print(f"  📝 {filename}...")
    content = llm(prompt)
    save(COMPANY_DIR / filename, content)

print("\n✅ All company pages generated!")

🏢 Generating company pages for Qafela (قافلة)...

  📝 about.md...
  ✅ Saved: knowledge-base/company/about.md
  📝 overview.md...
  ✅ Saved: knowledge-base/company/overview.md
  📝 culture.md...
  ✅ Saved: knowledge-base/company/culture.md
  📝 careers.md...
  ✅ Saved: knowledge-base/company/careers.md

✅ All company pages generated!


## Step 5 — Generate Employee Profiles

Each employee gets a full HR-style profile with:
- Summary (DOB, title, location, salary)
- Career progression (at the company + prior roles)
- Annual performance history
- Compensation history
- HR notes

In [13]:
EMPLOYEE_SYSTEM = (
    "You are an HR document writer for a tech company. "
    "Write realistic, detailed employee profiles in markdown format only. "
    "Output only the markdown. Do not add any commentary, preamble, or explanation."
)

def employee_prompt(emp):
    joined_year = random.randint(2015, 2022)
    years_range = ", ".join(str(y) for y in range(joined_year, 2026))
    return f"""
Write an HR employee profile for {emp['name']} at {c['name']}.
Title: {emp['title']}. Department: {emp['dept']}. Joined: {joined_year}.

Use EXACTLY these markdown sections:

# {emp['name']}

## Summary
- Date of Birth: (realistic for career stage)
- Job Title: {emp['title']}
- Location: (Cairo, New Cairo, Giza, Alexandria, or Remote in Egypt)
- Current Salary: (realistic Egyptian salary in EGP for {emp['title']})

## {c['name']} Career Progression
- {joined_year} - Present: {emp['title']} at {c['name']} — key achievements
- (2-3 previous roles at Egyptian companies, banks, or insurers)

## Annual Performance History
One bullet for each year: {years_range}
Format each as: - **YEAR**: **Rating** — one sentence reason.
Ratings: Exceptional / Exceeds Expectations / Meets Expectations / Satisfactory / Developing / Below Expectations

## Compensation History
One bullet for each year from {joined_year} to 2025.
Format: - **YEAR**: EGP X base + EGP Y bonus

## Other HR Notes
- Professional development
- Work-life balance
- Community engagement
"""

print(f"👥 Generating {len(c['employee_roles'])} employee profiles...\n")
for emp in c['employee_roles']:
    if should_skip(EMPLOYEES_DIR / f"{emp['name']}.md"):
        continue
    print(f"  📝 {emp['name']} ({emp['title']})...")
    content = llm(employee_prompt(emp), system=EMPLOYEE_SYSTEM)
    save(EMPLOYEES_DIR / f"{emp['name']}.md", content)

print(f"\n✅ All employee profiles generated!")

👥 Generating 32 employee profiles...

  📝 Mohamed Elbahluan (Co-Founder & CEO)...
  ✅ Saved: knowledge-base/employees/Mohamed Elbahluan.md
  📝 Mariam Hassan (Chief Technology Officer)...
  ✅ Saved: knowledge-base/employees/Mariam Hassan.md
  📝 Ahmed Fathy (Chief Financial Officer)...
  ✅ Saved: knowledge-base/employees/Ahmed Fathy.md
  📝 Nour El-Din Farouk (General Counsel)...
  ✅ Saved: knowledge-base/employees/Nour El-Din Farouk.md
  📝 Hana Abdelrahman (VP of People)...
  ✅ Saved: knowledge-base/employees/Hana Abdelrahman.md
  📝 Karim Mostafa (VP of Sales)...
  ✅ Saved: knowledge-base/employees/Karim Mostafa.md
  📝 Salma Ibrahim (Head of Product)...
  ✅ Saved: knowledge-base/employees/Salma Ibrahim.md
  📝 Omar Sherif (VP of Engineering)...
  ✅ Saved: knowledge-base/employees/Omar Sherif.md
  📝 Mohamed Tarek (Senior Software Engineer)...
  ✅ Saved: knowledge-base/employees/Mohamed Tarek.md
  📝 Layla Nabil (Backend Engineer)...
  ✅ Saved: knowledge-base/employees/Layla Nabil.md
  📝 Has

## Step 6 — Generate Product Spec Sheets

Each product gets a detailed spec sheet with:
- Summary (what it does, who it's for)
- Key features (7+)
- Pricing tiers (Basic / Professional / Enterprise)
- 2025-2026 product roadmap

In [ ]:
PRODUCT_SYSTEM = (
    "You are a B2B SaaS product marketing writer. "
    "Write detailed, realistic product specifications in markdown only. "
    "Output only the markdown. Do not add any commentary or preamble."
)

def product_prompt(product):
    return f"""
Write a product spec sheet for '{product['name']}' by {c['name']}.
Product type: {product['type']}
Company: {c['name']} — {c['description']}

Use EXACTLY these markdown sections:

# Product Summary
# {product['name']}

## Summary
2-3 paragraphs: what the product does, who it serves (B2B insurance carriers),
and what makes it unique. Mention AI and machine learning capabilities.

## Features
List at least 7 features. Each: **Feature Name**: 1-2 sentence description.
Features must be specific to {product['type']}.

## Pricing
- **Basic Tier**: $X/month — 3-4 bullet inclusions
- **Professional Tier**: $Y/month — 4-5 bullet inclusions
- **Enterprise Tier**: $Z/month — 5-6 bullet inclusions
Include a contact-sales note for custom pricing.

## 2025-2026 Roadmap
### Q1 2025: (title) — 2-3 bullet features
### Q2 2025: (title) — 2-3 bullet features
### Q3 2025: (title) — 2-3 bullet features
### Q4 2025: (title) — 2-3 bullet features
### 2026: (annual goals) — 2-3 bullet features
"""

print(f"📦 Generating {len(c['products'])} product spec sheets...\n")
for product in c['products']:
    if should_skip(PRODUCTS_DIR / f"{product['name']}.md"):
        continue
    print(f"  📝 {product['name']} ({product['type']})...")
    content = llm(product_prompt(product), system=PRODUCT_SYSTEM)
    save(PRODUCTS_DIR / f"{product['name']}.md", content)

print(f"\n✅ All product spec sheets generated!")

📦 Generating 8 product spec sheets...

  📝 Carllm (Auto Insurance Platform)...
  ✅ Saved: knowledge-base/products/Carllm.md
  📝 Homellm (Home Insurance Platform)...
  ✅ Saved: knowledge-base/products/Homellm.md
  📝 Healthllm (Health Insurance Platform)...
  ✅ Saved: knowledge-base/products/Healthllm.md
  📝 Lifellm (Life Insurance Platform)...


## Step 7 — Generate Client Contracts

For each product, generates `clients_per_product` contracts with fictional insurance companies matched to the product they buy (`CLIENT_POOL`).

In [ ]:
CONTRACT_SYSTEM = (
    "You are a legal and business document writer specializing in B2B SaaS contracts. "
    "Write detailed, realistic contracts in markdown only. "
    "Output only the markdown. Do not add any commentary or explanation."
)

# Each client is matched to the product it buys, so contracts make sense
# (an auto insurer buys Carllm, a health insurer buys Healthllm, ...).
CLIENT_POOL = [
    {"name": "Pharaoh Auto Insurance",        "contact": "Wael Sabry",    "title": "COO",               "product": "Carllm"},
    {"name": "Memphis Auto Group",            "contact": "Nader Lotfi",   "title": "COO",               "product": "Carllm"},
    {"name": "Sahara Home Insurance",         "contact": "Nermin Adel",   "title": "CEO",               "product": "Homellm"},
    {"name": "Giza Property Insurance",       "contact": "Marwa Essam",   "title": "Managing Director", "product": "Homellm"},
    {"name": "Oasis Health Plans",            "contact": "Reem Khalil",   "title": "CEO",               "product": "Healthllm"},
    {"name": "Rosetta Health Co.",            "contact": "Adel Fawzy",    "title": "President",         "product": "Healthllm"},
    {"name": "Delta Life Assurance",          "contact": "Hany Fouad",    "title": "CEO",               "product": "Lifellm"},
    {"name": "Lotus Life Assurance",          "contact": "Bassem Nour",   "title": "President",         "product": "Lifellm"},
    {"name": "Pyramids Commercial Insurance", "contact": "Inas Farag",    "title": "CEO",               "product": "Bizllm"},
    {"name": "Thebes Business Underwriters",  "contact": "Hala Samir",    "title": "CEO",               "product": "Bizllm"},
    {"name": "Sinai Claims Network",          "contact": "Hatem Gaber",   "title": "COO",               "product": "Claimllm"},
    {"name": "CairoCover Insurance",          "contact": "Laila Hamdy",   "title": "Managing Director", "product": "Claimllm"},
    {"name": "Red Sea Reinsurance",           "contact": "Tamer Hosny",   "title": "Managing Director", "product": "Rellm"},
    {"name": "NileShield Insurance",          "contact": "Amr El-Shamy",  "title": "CEO",               "product": "Rellm"},
    {"name": "Alexandria Medical Plans",      "contact": "Dalia Naguib",  "title": "President",         "product": "Markellm"},
    {"name": "Karnak Life Partners",          "contact": "Sherine Magdy", "title": "CEO",               "product": "Markellm"},
]

def contract_prompt(product, client, contract_num, year, tier, months):
    return f"""
Write a B2B SaaS contract between {c['name']} (an Egyptian company in Cairo) and {client['name']} (an Egyptian insurer). Quote all money in EGP.

Use EXACTLY this markdown structure:

# Contract with {client['name']} for {product['name']}

**Contract Date:** (realistic date in {year})
**Contract Number:** {contract_num}
**Parties:**
- {c['name']}, Cairo, Egypt
- {client['name']}

---

## Terms
1. Coverage: {tier} Tier of {product['name']} for {client['name']}
2. Duration: {months} months from contract date
3. Payment: specific monthly fee and total contract value (realistic for {tier} tier {product['type']} SaaS)
4. Policy Volume: number of policies/users {client['name']} manages
5. Service Level Agreement: uptime % and credit terms
6. Confidentiality: NDA clause
7. Data Security: SOC 2 and Egypt's Personal Data Protection Law (Law 151/2020)

---

## Renewal
Notice period, renewal options, pricing guarantee.

---

## Features
5-7 numbered features of the {tier} Tier of {product['name']} specific to {product['type']}.

---

## Support
1. Dedicated success team
2. Response SLAs (P1/P2/P3)
3. Implementation timeline
4. Training program

---

**Signatures:**

_________________________________
**{c['founder']}** | CEO | **{c['name']}** | Date: (contract date)

_________________________________
**{client['contact']}** | {client['title']} | **{client['name']}** | Date: (contract date)
"""

n_contracts = COMPANY['clients_per_product'] * len(c['products'])
print(f"📝 Generating {n_contracts} contracts ({COMPANY['clients_per_product']} per product × {len(c['products'])} products)...\n")

contract_idx = 1

for product in c['products']:
    product_clients = [cl for cl in CLIENT_POOL if cl["product"] == product["name"]]
    if len(product_clients) < c['clients_per_product']:
        raise ValueError(
            f"CLIENT_POOL has {len(product_clients)} clients for {product['name']}, "
            f"need {c['clients_per_product']}"
        )
    for cl in product_clients[:c['clients_per_product']]:
        year         = random.choice([2023, 2024, 2025])
        tier         = random.choice(["Basic", "Professional", "Enterprise"])
        months       = random.choice([12, 24, 36])
        contract_num = f"CR-{year}-{chr(65 + (contract_idx % 26))}-{contract_idx:04d}"
        contract_idx += 1

        filename = f"Contract with {cl['name']} for {product['name']}.md"
        if should_skip(CONTRACTS_DIR / filename):
            continue

        print(f"  📝 {cl['name']} × {product['name']} ({tier}, {months}mo)...")
        content = llm(contract_prompt(product, cl, contract_num, year, tier, months),
                      system=CONTRACT_SYSTEM)
        save(CONTRACTS_DIR / filename, content)

print(f"\n✅ All contracts generated!")

## Step 8 — Summary

In [ ]:
KB = Path("..").resolve() / "knowledge-base"

print("📁 Knowledge Base Summary")
print("=" * 45)

total_files = 0
total_bytes = 0
for folder in sorted(KB.iterdir()):
    if folder.is_dir():
        files = list(folder.rglob("*.md"))
        total_files += len(files)
        folder_bytes = sum(f.stat().st_size for f in files)
        total_bytes  += folder_bytes
        print(f"\n📂 {folder.name}/ ({len(files)} files, {folder_bytes/1024:.1f} KB)")
        for f in sorted(files):
            print(f"   • {f.name}")

print(f"\n{'=' * 45}")
print(f"Total : {total_files} files | {total_bytes/1024:.1f} KB")
print(f"\n🎉 Knowledge base is ready!")
print(f"   → Next: Open notebooks/02_rag_implementations.ipynb")

## Step 9 — Preview Sample Output

In [ ]:
KB = Path("..").resolve() / "knowledge-base"

for folder_name in ["company", "employees", "products", "contracts"]:
    folder = KB / folder_name
    files  = sorted(f for f in folder.glob("*.md") if f.name != ".gitkeep")
    if not files:
        print(f"\n⚠️  No files in {folder_name}/")
        continue
    sample  = files[0]
    content = sample.read_text(encoding="utf-8")
    print(f"\n{'═' * 60}")
    print(f"📄 {folder_name}/{sample.name}")
    print(f"{'═' * 60}")
    print(content[:700])
    if len(content) > 700:
        print(f"\n... ({len(content)-700} more chars)")

## Step 10 — Generate the evaluation test set

Run this **after** the knowledge-base markdown files exist.

```
one to four .md files (depends on the category)
    → Llama writes up to 4 questions from that text only
    → keep the question only if every keyword is inside the file
    → append one JSON line to evaluation/tests.jsonl
    → next file
```

Each line matches the evaluator in notebook 02:

`question`, `keywords`, `reference_answer`, `category`.

The target mix (150 questions):

| Category | Count | What it asks |
|---|---|---|
| `direct_fact` | 70 | One fact from a single document |
| `temporal` | 20 | A date, year, or duration |
| `spanning` | 20 | An answer that needs more than one document |
| `comparative` | 10 | A comparison of two amounts, people, or products |
| `numerical` | 10 | A count, salary, or price |
| `relationship` | 10 | How a person, product, or client is connected |
| `holistic` | 10 | A broad question about the company |

In [21]:
import json
import re
from pydantic import BaseModel, Field

class TestQuestion(BaseModel):
    question: str = Field(description="The question to ask the RAG system")
    keywords: list[str] = Field(description="Keywords that must appear in retrieved context")
    reference_answer: str = Field(description="The reference answer for this question")
    category: str = Field(description="direct_fact, temporal, spanning, comparative, numerical, relationship, or holistic")

class TestBatch(BaseModel):
    tests: list[TestQuestion]

EVAL_DIR = ROOT / "evaluation"
EVAL_DIR.mkdir(exist_ok=True)
TESTS_PATH = EVAL_DIR / "tests.jsonl"

# Target mix: 150 questions.
CATEGORY_TARGETS = {
    "direct_fact": 70,
    "temporal": 20,
    "spanning": 20,
    "comparative": 10,
    "numerical": 10,
    "relationship": 10,
    "holistic": 10,
}
CATEGORY_HELP = {
    "direct_fact": "Ask for one explicit fact stated in a single document, such as a job title, location, previous employer, or product name. Do not ask about salary, bonus, payment, price, duration, dates, or counts.",
    "temporal": "Ask when something happened, started, or how long it lasts. The answer must include a date, year, quarter, or duration from the text.",
    "spanning": "Ask a question whose answer requires facts from more than one of the documents provided.",
    "comparative": "Compare two amounts, people, products, or contracts that appear in the documents.",
    "numerical": "Ask for a count, salary, price, or contract total. The answer must include the number from the text, in EGP when the text uses EGP.",
    "relationship": "Ask how a person, product, client, or company is connected. Examples: who signed, who leads, which client uses which product.",
    "holistic": "Ask a broad question about Qafela (قافلة) as a company. The answer should draw on several parts of the documents.",
}

def extract_json(text: str) -> str:
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1:
        raise ValueError("No JSON object in model reply")
    return text[start:end + 1]

def pick_sources(category: str, files: list[Path]) -> list[Path]:
    by_folder = {}
    for path in files:
        by_folder.setdefault(path.parent.name, []).append(path)
    if category == "holistic":
        return by_folder.get("company", files)[:4]
    if category == "numerical":
        pool = by_folder.get("contracts", []) + by_folder.get("products", []) + by_folder.get("employees", [])
        return [random.choice(pool or files)]
    if category == "relationship":
        pool = by_folder.get("employees", []) + by_folder.get("contracts", [])
        return random.sample(pool or files, k=min(2, len(pool or files)))
    if category in {"spanning", "comparative"}:
        return random.sample(files, k=min(2, len(files)))
    return [random.choice(files)]

GENERIC_KEYWORDS = {
    "job title", "location", "current salary", "salary", "payment", "duration",
    "contract date", "contract number", "policy volume", "notice period",
    "company name", "core value", "core values", "the contract", "the company",
}

COMPANY_ONLY = {"qafela", "\u0642\u0627\u0641\u0644\u0629", "qafela (\u0642\u0627\u0641\u0644\u0629)"}

def norm(text: str) -> str:
    return re.sub(r"\s+", " ", text.lower()).strip()

PARTY_RE = r"Qafela|Carllm|Homellm|Healthllm|Lifellm|Bizllm|Claimllm|Rellm|Markellm|[A-Z][a-z]+(?:\s+[A-Z][a-z]+)+"

def named_parties(question: str) -> list:
    return re.findall(
        PARTY_RE,
        question,
    )

def fact_key_for(test: TestQuestion) -> str:
    parties = " ".join(dict.fromkeys(named_parties(test.question)))
    return "fact:" + norm(parties + " " + test.reference_answer)

def keyword_is_specific(keyword: str) -> bool:
    if norm(keyword) in GENERIC_KEYWORDS or len(keyword.strip()) < 3:
        return False
    if re.search(r"\d", keyword):
        return True
    if re.search(r"qafela|llm|\begp\b|cr-", keyword, re.I):
        return True
    return len(keyword.split()) >= 2

def salvage_keywords(test: TestQuestion, combined: str) -> None:
    original = list(test.keywords)
    kept = []
    for keyword in test.keywords:
        if norm(keyword) in GENERIC_KEYWORDS or keyword.lower() not in combined.lower():
            continue
        kept.append(keyword.strip())
    answer = " ".join(test.reference_answer.split())
    candidates = []
    if 2 < len(answer) <= 80:
        candidates.append(answer)
    candidates.extend(
        re.findall(
            r"CR-[\w-]+|(?:EGP\s*)?\d[\d,]*(?:\.\d+)?(?:\s*EGP)?|\b(?:19|20)\d{2}\b",
            answer,
        )
    )
    for candidate in candidates:
        token = candidate.strip(" .")
        if not token or norm(token) in GENERIC_KEYWORDS:
            continue
        if token.lower() not in combined.lower() or token in kept:
            continue
        if any(token.lower() in existing.lower() for existing in kept):
            continue
        kept.append(token)
    deduped = []
    for keyword in kept:
        if keyword not in deduped:
            deduped.append(keyword)
    test.keywords = deduped
    if deduped != original:
        print(f"    keywords {original} -> {deduped}")

def reject_reason(test: TestQuestion, combined: str, seen: set):
    question = test.question.strip()
    question_key = norm(question)
    if question_key in seen or fact_key_for(test) in seen:
        return "duplicate"
    if test.keywords and all(norm(keyword) in COMPANY_ONLY for keyword in test.keywords):
        return "keyword is only the company name"
    if not test.keywords:
        return "no fact keyword left in the source"
    answer_text = test.reference_answer.lower()
    useful = [
        keyword
        for keyword in test.keywords
        if keyword_is_specific(keyword) or keyword.lower() in answer_text
    ]
    if not useful:
        return "keyword is a heading, not the fact"
    names_subject = re.search(
        PARTY_RE,
        question,
    )
    if test.category != "holistic" and not names_subject:
        return "question does not name the person, product, or client"
    blob = f"{question} {test.reference_answer}".lower()
    if test.category == "temporal" and not re.search(r"\d{4}|month|year|when|how long", blob):
        return "not temporal"
    if test.category == "numerical" and not re.search(r"\d", test.reference_answer):
        return "answer has no number"
    if test.category == "comparative" and not re.search(r"compar|than|versus|\bvs\b|difference|higher|lower", question.lower()):
        return "not a comparison"
    if test.category == "relationship":
        parties = re.findall(
            PARTY_RE,
            question,
        )
        linked = re.search(r"signed|leads|works with|between|reports", question.lower())
        if len(parties) < 2 and not linked:
            return "not a relationship"
    if test.category == "direct_fact" and re.search(
        r"\b(when|how long|how much|how many|salary|bonus|payment|price|cost|duration)\b|what is the year|which year",
        question.lower(),
    ):
        return "belongs in temporal or numerical"
    return None

def questions_for(category: str, sources: list, how_many: int, seen: set) -> list:
    blobs = []
    for path in sources:
        blobs.append(f"\n\n# FILE: {path.parent.name}/{path.name}\n{path.read_text(encoding='utf-8')[:4000]}")
    combined = "".join(blobs)
    prompt = f"""
Write exactly {how_many} evaluation questions for a RAG system about Qafela (قافلة).
Category for every question: {category}
{CATEGORY_HELP[category]}

Rules:
- Use only facts written in the documents below. Do not invent names, dates, or amounts.
- Every keyword must appear verbatim in the documents.
- Keywords must be the fact itself: a person name, client name, product name, amount, date, or contract number.
- Never use a heading as a keyword. Forbidden keywords: Job Title, Location, Current Salary, Payment, Duration, Contract Date, Contract Number.
- The question must name the person, product, or client. Do not ask "the contract" or "the company" without a name.
- reference_answer must be answerable from these documents alone.
- Set category to "{category}" on every item.

Return JSON only:
{{"tests": [{{"question": "...", "keywords": ["..."], "reference_answer": "...", "category": "{category}"}}]}}

Documents:
{combined}
"""
    reply = client.chat.completions.create(
        model=MODEL_ID,
        messages=[
            {"role": "system", "content": "You write RAG evaluation items. Output JSON only. Use only facts from the documents."},
            {"role": "user", "content": prompt},
        ],
        temperature=0.2,
        max_tokens=MAX_TOKENS,
    ).choices[0].message.content
    batch = TestBatch.model_validate_json(extract_json(reply))
    kept = []
    for test in batch.tests:
        test.category = category
        salvage_keywords(test, combined)
        reason = reject_reason(test, combined, seen)
        if reason:
            print(f"    skipped ({reason}): {test.question[:70]}")
            continue
        seen.add(norm(test.question))
        seen.add(fact_key_for(test))
        kept.append(test)
    return kept[:how_many]

files = sorted(path for path in KB.rglob("*.md") if path.name != ".gitkeep")
if not files:
    raise FileNotFoundError("No markdown files in knowledge-base/. Run Steps 4–7 first.")

if TESTS_PATH.exists() and not REBUILD_TESTS:
    print(f"⏭️  {TESTS_PATH.relative_to(ROOT)} exists — skipping (set REBUILD_TESTS = True to regenerate)")
else:
    random.seed(7)
    tests = []
    seen = set()
    print(f"Building {sum(CATEGORY_TARGETS.values())} tests from {len(files)} documents\n")
    for category, target in CATEGORY_TARGETS.items():
        got = []
        attempts = 0
        print(f"{category}: target {target}")
        while len(got) < target and attempts < target * 4:
            attempts += 1
            need = min(4, target - len(got))
            sources = pick_sources(category, files)
            names = ", ".join(f"{path.parent.name}/{path.name}" for path in sources)
            print(f"  attempt {attempts} ({need} from {names})")
            try:
                got.extend(questions_for(category, sources, need, seen))
            except Exception as exc:
                print(f"    failed: {exc}")
        got = got[:target]
        tests.extend(got)
        print(f"  kept {len(got)}/{target}\n")

    with TESTS_PATH.open("w", encoding="utf-8") as handle:
        for test in tests:
            handle.write(test.model_dump_json() + "\n")

    print(f"Wrote {len(tests)} tests to {TESTS_PATH.relative_to(ROOT)}")
    for category, target in CATEGORY_TARGETS.items():
        found = sum(1 for test in tests if test.category == category)
        print(f"  {category}: {found} (target {target})")

Building 150 tests from 60 documents

direct_fact: target 70
  attempt 1 (4 from employees/Ahmed Fathy.md)
    keywords ['Job Title'] -> ['Chief Financial Officer']
    keywords ['Location'] -> ['Cairo, New Cairo']
    keywords ['Product Name'] -> ['ERP system']
    keywords ['Client Name'] -> []
    skipped (no fact keyword left in the source): What is the name of the local charity that Ahmed Fathy serves as a boa
  attempt 2 (4 from contracts/Contract with Lotus Life Assurance for Markellm.md)
    keywords ['Qafela', 'Cairo', 'Egypt'] -> ['Qafela', 'Cairo', 'Egypt', 'Qafela (قافلة), Cairo, Egypt']
  attempt 3 (4 from employees/Farida Adel.md)
    keywords ['Job Title'] -> ['Product Manager']
    keywords ['Location'] -> ['Bank of Alexandria']
    keywords ['Company Name'] -> ['Qafela (قافلة)']
    skipped (keyword is only the company name): What is the name of the company where Farida Adel worked as a Product 
    keywords ['Product Line'] -> ['Product Line', "Qafela's new product li